# 03 - Neo4j Ingestion: Entity Extraction → Knowledge Graph

Extracts entities from Weaviate chunks via Groq → populates Neo4j.

In [1]:
import os, json, re, time
from dotenv import load_dotenv

load_dotenv()

GROQ_API_KEY     = os.getenv('GROQ_API_KEY')
GROQ_MODEL       = 'llama-3.3-70b-versatile'
NEO4J_URI        = os.getenv('NEO4J_URI')
NEO4J_USERNAME   = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD   = os.getenv('NEO4J_PASSWORD')
NEO4J_DATABASE   = os.getenv('NEO4J_DATABASE', 'neo4j')
HF_API_KEY       = os.getenv('HF_API_KEY')
COHERE_API_KEY   = os.getenv('COHERE_API_KEY', '')
WEAVIATE_URL     = os.getenv('WEAVIATE_URL', '').rstrip('/')
WEAVIATE_API_KEY = os.getenv('WEAVIATE_API_KEY')
COLLECTION       = 'ScmChunks'
MAX_CHUNKS       = 80

print(f'LLM   : {GROQ_MODEL}')
print(f'Neo4j : {NEO4J_URI}')

LLM   : llama-3.3-70b-versatile
Neo4j : neo4j+s://f7a4bcd2.databases.neo4j.io


## 1. Test Neo4j

In [2]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
with driver.session(database=NEO4J_DATABASE) as s:
    print(s.run("RETURN 'Connected' AS m").single()['m'])
    print(f'Existing nodes: {s.run("MATCH (n) RETURN count(n) AS c").single()["c"]}')

Connected
Existing nodes: 7297


## 2. Load chunks from Weaviate

In [3]:
import weaviate
from weaviate.classes.init import Auth, AdditionalConfig, Timeout

_headers = {}
if HF_API_KEY:
    _headers['X-HuggingFace-Api-Key'] = HF_API_KEY.strip()
if COHERE_API_KEY:
    _headers['X-Cohere-Api-Key'] = COHERE_API_KEY.strip()

wv = weaviate.connect_to_weaviate_cloud(
    cluster_url      = WEAVIATE_URL,
    auth_credentials = Auth.api_key(WEAVIATE_API_KEY),
    headers          = _headers,
    additional_config= AdditionalConfig(timeout=Timeout(init=60, query=30, insert=60)),
    skip_init_checks = True,
)

collection = wv.collections.get(COLLECTION)

chunks = []
for obj in collection.iterator(return_properties=['chunk_id', 'source_file', 'text']):
    chunks.append(obj.properties)
    if len(chunks) >= MAX_CHUNKS: break
wv.close()
print(f'Loaded {len(chunks)} chunks')

Loaded 80 chunks


## 3. Entity extraction

In [4]:
from groq import Groq

groq_client = Groq(api_key=GROQ_API_KEY)

PROMPT = (
    'Extract supply chain entities and relationships.\n'
    'Return ONLY valid JSON (no markdown):\n'
    '{{"entities":[{{"id":"e1","name":"...","type":"Supplier|Product|Location|Organization|Process|Risk|Metric|Policy|Technology"}}],'
    ' "relationships":[{{"source":"e1","target":"e2","relation":"SUPPLIES_TO|LOCATED_IN|PART_OF|DEPENDS_ON|MANAGES|AFFECTS|PRODUCES|REQUIRES|SHIPS_TO|MITIGATES|OPTIMIZES"}}]}}\n'
    'Text: {text}'
)
_reg = {}

def resolve(name):
    key = re.sub(r'\s+', ' ', name.lower().strip())
    if key not in _reg: _reg[key] = name
    return _reg[key]

def parse_json(raw):
    raw = re.sub(r'```(?:json)?', '', raw).strip()
    m = re.search(r'\{.*\}', raw, re.DOTALL)
    if m:
        try: return json.loads(m.group())
        except: pass
    return {'entities': [], 'relationships': []}

def extract(text):
    try:
        r = groq_client.chat.completions.create(
            model=GROQ_MODEL,
            messages=[{'role': 'user', 'content': PROMPT.format(text=text)}],
            temperature=0, max_tokens=800,
        )
        return parse_json(r.choices[0].message.content)
    except Exception as e:
        print(f'  warn: {e}'); return {'entities': [], 'relationships': []}

print('Extractor ready')

Extractor ready


## 4. Neo4j writer

In [5]:
def write(driver, ext, source):
    with driver.session(database=NEO4J_DATABASE) as s:
        id2n = {}
        for ent in ext.get('entities', []):
            raw = ent['name'].strip()
            if not raw: continue
            name = resolve(raw); id2n[ent['id']] = name
            s.run('MERGE (n:Entity {name:$n}) SET n.type=$t, n.source=$s, n.updated_at=datetime()',
                  n=name, t=ent.get('type','Unknown'), s=source)
        for rel in ext.get('relationships', []):
            src = id2n.get(rel['source'],'')
            tgt = id2n.get(rel['target'],'')
            if not src or not tgt: continue
            rt = rel.get('relation','RELATED_TO')
            s.run(f'MATCH (a:Entity{{name:$s}}) MATCH (b:Entity{{name:$t}}) MERGE (a)-[r:{rt}]->(b) SET r.source=$src',
                  s=src, t=tgt, src=source)
print('Writer ready')

Writer ready


## 5. Run pipeline

> **Idempotent**: Neo4j uses `MERGE` so re-running never creates duplicates.
> Driver is reconnected here to avoid `SessionExpired` from idle timeout between cells.

In [6]:
# Reconnect — AuraDB drops idle connections between cells
driver.close()
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

te = tr = 0
for i, chunk in enumerate(chunks):
    ext = extract(chunk['text'])
    write(driver, ext, chunk.get('source_file','?'))
    te += len(ext.get('entities',[]))
    tr += len(ext.get('relationships',[]))
    if (i+1) % 10 == 0: print(f'  [{i+1}/{len(chunks)}] entities={te} rels={tr}')
    time.sleep(0.3)
print(f'Done: {te} entities, {tr} relationships')

  [10/80] entities=80 rels=80
  [20/80] entities=144 rels=138
  [30/80] entities=237 rels=219
  [40/80] entities=303 rels=292
  [50/80] entities=385 rels=372
  [60/80] entities=484 rels=451
  [70/80] entities=571 rels=531
  [80/80] entities=654 rels=602
Done: 654 entities, 602 relationships


## 6. Validate

In [7]:
with driver.session(database=NEO4J_DATABASE) as s:
    print(f'Nodes : {s.run("MATCH (n) RETURN count(n) AS c").single()["c"]}')
    print(f'Edges : {s.run("MATCH ()-[r]->() RETURN count(r) AS c").single()["c"]}')
    print('\nTop types:')
    for r in s.run('MATCH (n) RETURN n.type AS t, count(n) AS c ORDER BY c DESC LIMIT 8'):
        print(f'  {str(r["t"] or "Unknown"):<22} {r["c"]}')
    print('\nSample triples:')
    for r in s.run('MATCH (a)-[r]->(b) RETURN a.name,type(r),b.name LIMIT 5'):
        print(f'  {r["a.name"]} --[{r["type(r)"]}]--> {r["b.name"]}')
driver.close()

Nodes : 7808
Edges : 17045

Top types:
  Unknown                7297
  Organization           131
  Technology             115
  Process                98
  Metric                 48
  Location               38
  Risk                   27
  Product                26

Sample triples:
  None --[IN_REGION]--> None
  None --[IN_REGION]--> None
  None --[IN_REGION]--> None
  None --[IN_REGION]--> None
  None --[IN_REGION]--> None
